<a href="https://www.kaggle.com/code/samithsachidanandan/rogii-wellbore-geology-prediction-ensemble?scriptVersionId=317128221" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Ensemble: LightGBM + XGBoost + TCN-NN
## ROGII Wellbore Geology Prediction


## 0. Imports & Configuration

In [ ]:
from __future__ import annotations
from pathlib import Path
from collections import defaultdict
import copy, gc, os, random, time, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)

# ── Shared ────────────────────────────────────────────────────────────────────
RANDOM_STATE    = 42
FAST_DEBUG      = bool(int(os.environ.get('FAST_DEBUG', '0')))
N_FOLDS         = 2   if FAST_DEBUG else 5
MAX_TRAIN_WELLS = 24  if FAST_DEBUG else None

# ── LightGBM ──────────────────────────────────────────────────────────────────
OFFSETS    = np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80], dtype=np.float32)
LGB_PARAMS = dict(
    n_estimators     = 80  if FAST_DEBUG else 5000,
    learning_rate    = 0.059976685297931195,
    num_leaves       = 89,
    min_child_samples= 10,
    min_child_weight = 0.5425237767880097,
    subsample        = 0.6452823633939004,
    subsample_freq   = 1,
    colsample_bytree = 0.8213924491907012,
    reg_lambda       = 87.27971117911044,
    reg_alpha        = 2.0325709613371545,
    objective        = 'regression',
    random_state     = RANDOM_STATE,
    force_row_wise   = True,
    verbosity        = -1,
    n_jobs           = -1,
)


# ── XGBoost ───────────────────────────────────────────────────────────────────
XGB_PARAMS = dict(
    n_estimators     = 80  if FAST_DEBUG else 450,
    learning_rate    = 0.06 if FAST_DEBUG else 0.035,
    max_depth        = 5,
    min_child_weight = 20,
    subsample        = 0.85,
    colsample_bytree = 0.85,
    reg_lambda       = 4.0,
    reg_alpha        = 0.05,
    objective        = 'reg:squarederror',
    eval_metric      = 'rmse',
    tree_method      = 'hist',
    max_bin          = 256,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    device           = 'cuda' if __import__('torch').cuda.is_available() else 'cpu',
)

# ── Neural Network ────────────────────────────────────────────────────────────
EPOCHS        = 2   if FAST_DEBUG else 15
WARMUP_EPOCHS = 0   if FAST_DEBUG else 2
BATCH_SIZE    = 4   if FAST_DEBUG else 6
HIDDEN_SIZE   = 32  if FAST_DEBUG else 128
NUM_BLOCKS    = 3   if FAST_DEBUG else 6
LEARNING_RATE = 2e-3
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0
DROPOUT       = 0.10
HUBER_DELTA   = 1.0
TTA_ROUNDS    = 1   if FAST_DEBUG else 4
TTA_GR_NOISE  = 0.02

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

print(f'FAST_DEBUG={FAST_DEBUG} | DEVICE={DEVICE} | N_FOLDS={N_FOLDS} | NN_EPOCHS={EPOCHS}')


## 1. Data Root

In [ ]:
def find_data_root():
    candidates = [
        Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
        Path('/kaggle/input/rogii-wellbore-geology-prediction'),
        Path.cwd(),
    ]
    candidates.extend(Path.cwd().parents)
    for root in candidates:
        if (root / 'train').is_dir() and (root / 'sample_submission.csv').is_file():
            return root.resolve()
    raise FileNotFoundError('Cannot find data root')

DATA_ROOT = find_data_root()
TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR  = DATA_ROOT / 'test'
SAMPLE_SUB_PATH = DATA_ROOT / 'sample_submission.csv'

def well_id_from_path(p):
    return Path(p).name.split('__', 1)[0]

train_horizontal_paths = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
test_horizontal_paths  = sorted(TEST_DIR.glob('*__horizontal_well.csv'))
if MAX_TRAIN_WELLS:
    train_horizontal_paths = train_horizontal_paths[:MAX_TRAIN_WELLS]

print(f'Train: {len(train_horizontal_paths)} wells | Test: {len(test_horizontal_paths)} wells')


## 2. Shared Low-Level Helpers


In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true, np.float64) - np.asarray(y_pred, np.float64))**2)))

def robust_slope(x, y, default=0.0):
    x, y = np.asarray(x, np.float64), np.asarray(y, np.float64)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2 or np.nanstd(x[m]) < 1e-6: return default
    return float(np.polyfit(x[m], y[m], 1)[0])

def safe_interp(x, xp, fp):
    x, xp, fp = np.asarray(x, np.float64), np.asarray(xp, np.float64), np.asarray(fp, np.float64)
    m = np.isfinite(xp) & np.isfinite(fp)
    if m.sum() < 2: return np.full_like(x, np.nan)
    o = np.argsort(xp[m])
    return np.interp(x, xp[m][o], fp[m][o], left=np.nan, right=np.nan)

def rolling_quantile(series, window, q, min_periods=5):
    return series.rolling(window=window, center=True, min_periods=min_periods).quantile(q).bfill().ffill().fillna(0).to_numpy(dtype=float)

def load_typewell(well_id, split):
    p = (TRAIN_DIR if split == 'train' else TEST_DIR) / f'{well_id}__typewell.csv'
    return pd.read_csv(p) if p.exists() else pd.DataFrame({'TVT': [], 'GR': []})

# ── LightGBM beam helpers (from original notebook) ───────────────────────────
def nearest_index(sorted_values, target):
    idx = int(np.searchsorted(sorted_values, target, side='left'))
    if idx >= len(sorted_values): return len(sorted_values) - 1
    if idx > 0 and abs(sorted_values[idx-1]-target) <= abs(sorted_values[idx]-target): return idx-1
    return idx

def fill_and_smooth_gr(values, fallback, radius):
    s = pd.Series(values, dtype='float32').interpolate(limit_direction='both').fillna(fallback)
    if radius <= 0: return s.to_numpy(dtype=np.float32)
    return s.rolling(radius*2+1, center=True, min_periods=1).mean().to_numpy(dtype=np.float32)

def beam_predict(gr_values, tw_tvt, tw_gr, start_tvt, beam_size, move_cost, emit_scale, radius):
    start_idx   = nearest_index(tw_tvt, start_tvt)
    smoothed_gr = fill_and_smooth_gr(gr_values, float(np.nanmean(tw_gr)), radius)
    states, backpointers = {start_idx: 0.0}, []
    for gr_val in smoothed_gr:
        candidates, parents = {}, {}
        for idx, cost in states.items():
            for delta in (-1, 0, 1):
                ni = idx + delta
                if ni < 0 or ni >= len(tw_tvt): continue
                total = cost + ((gr_val - tw_gr[ni])**2) / emit_scale + move_cost * abs(delta)
                if ni not in candidates or total < candidates[ni]:
                    candidates[ni] = total; parents[ni] = idx
        kept = sorted(candidates.items(), key=lambda x: x[1])[:beam_size]
        states = dict(kept)
        backpointers.append({i: parents[i] for i, _ in kept})
    final_idx = min(states, key=states.get)
    path = [final_idx]
    for step in range(len(backpointers)-1, 0, -1):
        path.append(backpointers[step][path[-1]])
    path.reverse()
    return tw_tvt[np.asarray(path, dtype=np.int32)]


## 3. Feature Builders

Two separate builders are kept intentional:
- `build_lgb_features` — LightGBM's beam-search features (unique, expensive, **not** used by XGB/NN)
- `build_sequence` — shared feature array for XGB + NN (rolling GR stats, trajectory features, typewell interp)


In [ ]:


def build_lgb_features(horizontal_path, split='train'):
    well_id = well_id_from_path(horizontal_path)
    df  = pd.read_csv(horizontal_path)
    tw  = load_typewell(well_id, split)

    mask = df['TVT_input'].isna().to_numpy()
    if not mask.any(): return None
    mask_start = int(np.flatnonzero(mask)[0])
    if mask_start == 0: return None

    # Require typewell for beam features
    if not {'TVT', 'GR'}.issubset(tw.columns) or len(tw) < 2:
        return None

    known  = df.iloc[:mask_start].copy()
    hidden = df.iloc[mask_start:].copy()
    last_k = known.iloc[-1]

    tw_tvt = tw['TVT'].to_numpy(dtype=np.float32)
    tw_gr  = tw['GR'].to_numpy(dtype=np.float32)

    gr_full  = df['GR'].interpolate(limit_direction='both')
    if gr_full.isna().any(): gr_full = gr_full.fillna(float(np.nanmean(tw_gr)))

    gr_roll5  = gr_full.rolling(5,  center=True, min_periods=1).mean()
    gr_roll21 = gr_full.rolling(21, center=True, min_periods=1).mean()
    gr_grad   = gr_full.diff().fillna(0.0)

    known_tvt = known['TVT_input'].to_numpy(dtype=np.float32)
    known_md  = known['MD'].to_numpy(dtype=np.float32)
    known_z   = known['Z'].to_numpy(dtype=np.float32)

    prefix_tw_gr   = np.interp(known_tvt, tw_tvt, tw_gr)
    prefix_gr      = gr_full.iloc[:mask_start].to_numpy(dtype=np.float32)
    prefix_resid   = prefix_gr - prefix_tw_gr
    prefix_tw_rmse = float(np.sqrt(np.mean(prefix_resid**2)))
    prefix_tw_mae  = float(np.mean(np.abs(prefix_resid)))

    last_known_tvt = float(last_k['TVT_input'])
    hidden_gr      = hidden['GR'].to_numpy(dtype=np.float32)

    beam_cons  = beam_predict(hidden_gr, tw_tvt, tw_gr, last_known_tvt, 10, 20.0, 144.0, 2)
    beam_loose = beam_predict(hidden_gr, tw_tvt, tw_gr, last_known_tvt, 10,  8.0,  64.0, 2)

    hidden_gr_filled = gr_full.iloc[mask_start:].to_numpy(dtype=np.float32)
    offset_diffs = {
        f'tw_diff_{int(o)}': hidden_gr_filled - np.float32(np.interp(last_known_tvt+float(o), tw_tvt, tw_gr))
        for o in OFFSETS
    }

    def recent_mean_diff(vals, w):
        vals = vals[-(w+1):]
        return 0.0 if len(vals) < 2 else float(np.diff(vals).mean())

    def recent_slope_f(y, x, w):
        y, x = y[-w:], x[-w:]
        if len(y) < 2: return 0.0
        cx = x - x.mean()
        d = float(np.dot(cx, cx))
        return 0.0 if d == 0 else float(np.dot(cx, y - y.mean()) / d)

    features = pd.DataFrame({
        'well_id':         well_id,
        'row_index':       hidden.index.to_numpy(dtype=np.int32),
        'last_known_tvt':  np.float32(last_known_tvt),
        'known_len':       np.int32(mask_start),
        'hidden_len':      np.int32(len(hidden)),
        'frac_hidden':     ((hidden.index - mask_start) / max(len(hidden)-1, 1)).astype(np.float32),
        'md':              hidden['MD'].to_numpy(dtype=np.float32),
        'z':               hidden['Z'].to_numpy(dtype=np.float32),
        'x':               hidden['X'].to_numpy(dtype=np.float32),
        'y':               hidden['Y'].to_numpy(dtype=np.float32),
        'gr':              hidden_gr_filled,
        'gr_missing':      hidden['GR'].isna().to_numpy(dtype=np.int8),
        'gr_roll5':        gr_roll5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_roll21':       gr_roll21.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_grad':         gr_grad.iloc[mask_start:].to_numpy(dtype=np.float32),
        'dmd':             (hidden['MD'] - float(last_k['MD'])).to_numpy(dtype=np.float32),
        'dz':              (hidden['Z'] - float(last_k['Z'])).to_numpy(dtype=np.float32),
        'dx':              (hidden['X'] - float(last_k['X'])).to_numpy(dtype=np.float32),
        'dy':              (hidden['Y'] - float(last_k['Y'])).to_numpy(dtype=np.float32),
        'dist_xy':         np.sqrt((hidden['X']-float(last_k['X']))**2 + (hidden['Y']-float(last_k['Y']))**2).to_numpy(dtype=np.float32),
        'prefix_tvt_step20':      np.float32(recent_mean_diff(known_tvt, 20)),
        'prefix_tvt_step100':     np.float32(recent_mean_diff(known_tvt, 100)),
        'prefix_tvt_md_slope100': np.float32(recent_slope_f(known_tvt, known_md, 100)),
        'prefix_tvt_z_slope100':  np.float32(recent_slope_f(known_tvt, known_z, 100)),
        'prefix_tw_rmse':         np.float32(prefix_tw_rmse),
        'prefix_tw_mae':          np.float32(prefix_tw_mae),
        'beam_cons_delta':        (beam_cons  - np.float32(last_known_tvt)).astype(np.float32),
        'beam_loose_delta':       (beam_loose - np.float32(last_known_tvt)).astype(np.float32),
        'beam_gap':               (beam_loose - beam_cons).astype(np.float32),
    })

    for name, vals in offset_diffs.items():
        features[name] = vals.astype(np.float32)

    if split == 'train':
        features['target_tvt']      = hidden['TVT'].to_numpy(dtype=np.float32)
        features['target_residual'] = features['target_tvt'] - np.float32(last_known_tvt)

    return features.reset_index(drop=True)


In [ ]:


def build_sequence(horizontal_path, split='train', test_row_map=None):
    well_id = well_id_from_path(horizontal_path)
    h  = pd.read_csv(horizontal_path)
    tw = load_typewell(well_id, split)
    n  = len(h)
    row_index    = np.arange(n, dtype=np.int64)
    known_mask   = h['TVT_input'].notna().to_numpy()
    missing_mask = h['TVT_input'].isna().to_numpy()

    if split == 'train':
        target_mask = missing_mask & h['TVT'].notna().to_numpy()
    else:
        target_mask = missing_mask.copy()
        if test_row_map is not None and well_id in test_row_map:
            vi = test_row_map[well_id]; vi = vi[(vi>=0)&(vi<n)]
            target_mask = np.zeros(n, bool); target_mask[vi] = True

    known = h.loc[known_mask].copy()
    if len(known) == 0: known = h.head(1).copy(); known['TVT_input'] = np.nan

    first_missing = np.flatnonzero(missing_mask)
    ps_idx = int(first_missing[0]) if len(first_missing) else n
    lk = known.iloc[-1]

    ps_md = float(lk.get('MD', np.nan)); ps_x = float(lk.get('X', np.nan))
    ps_y  = float(lk.get('Y', np.nan));  ps_z = float(lk.get('Z', np.nan))
    ps_gr = float(lk.get('GR', np.nan)); last_known_tvt = float(lk.get('TVT_input', np.nan))

    slope_all    = robust_slope(known['MD'], known['TVT_input'])
    r200         = known.tail(min(200, len(known)))
    slope_recent = robust_slope(r200['MD'], r200['TVT_input'], default=slope_all)
    slope_z      = robust_slope(r200['Z'],  r200['TVT_input'])

    md = h['MD'].astype(float).to_numpy(); x = h['X'].astype(float).to_numpy()
    y  = h['Y'].astype(float).to_numpy();  z = h['Z'].astype(float).to_numpy()

    gr_raw    = h['GR'].astype(float)
    gr_filled = gr_raw.interpolate(limit_direction='both')
    gr_filled = gr_filled.fillna(gr_filled.mean() if not gr_filled.isna().all() else 0.0)

    tvt_ff = h['TVT_input'].astype(float).ffill().bfill().fillna(last_known_tvt)

    md_from_ps = md - ps_md; x_from_ps = x - ps_x; y_from_ps = y - ps_y; z_from_ps = z - ps_z
    xy_d  = np.sqrt(x_from_ps**2 + y_from_ps**2)
    xyz_d = np.sqrt(xy_d**2 + z_from_ps**2)
    baseline     = np.full(n, last_known_tvt, dtype=np.float64)
    bl_all       = last_known_tvt + slope_all    * md_from_ps
    bl_recent    = last_known_tvt + slope_recent * md_from_ps

    z_s = pd.Series(z); md_s = pd.Series(md)
    z10 = z_s.diff(10).fillna(0).to_numpy(float); md10 = md_s.diff(10).fillna(1).to_numpy(float)
    z50 = z_s.diff(50).fillna(0).to_numpy(float); md50 = md_s.diff(50).fillna(1).to_numpy(float)
    zs10 = np.where(np.abs(md10)>1e-6, z10/md10, 0.)
    zs50 = np.where(np.abs(md50)>1e-6, z50/md50, 0.)
    mc = np.cumsum(np.abs(np.diff(md, prepend=md[0])))
    mc_ps = mc - mc[ps_idx] if ps_idx < n else mc - mc[-1]

    fd = {
        'MD': md, 'X': x, 'Y': y, 'Z': z,
        'GR_filled': gr_filled.to_numpy(float), 'GR_missing': gr_raw.isna().astype(float).to_numpy(),
        'row_index': row_index.astype(float), 'row_frac': row_index/max(n-1,1),
        'known_tvt_mask': known_mask.astype(float),
        'TVT_input_filled': tvt_ff.to_numpy(float),
        'TVT_input_delta_last': tvt_ff.to_numpy(float) - last_known_tvt,
        'last_known_tvt': np.full(n, last_known_tvt),
        'known_tvt_range': np.full(n, known['TVT_input'].max()-known['TVT_input'].min()),
        'known_tvt_std': np.full(n, known['TVT_input'].std()),
        'known_gr_mean': np.full(n, gr_raw[known_mask].mean()),
        'known_gr_std':  np.full(n, gr_raw[known_mask].std()),
        'last_known_gr': np.full(n, ps_gr),
        'slope_tvt_md_all': np.full(n, slope_all),
        'slope_tvt_md_recent': np.full(n, slope_recent),
        'slope_tvt_z_recent': np.full(n, slope_z),
        'row_from_ps': row_index.astype(float)-ps_idx,
        'md_from_ps': md_from_ps, 'x_from_ps': x_from_ps, 'y_from_ps': y_from_ps, 'z_from_ps': z_from_ps,
        'xy_dist_from_ps': xy_d, 'xyz_dist_from_ps': xyz_d,
        'baseline_tvt': baseline, 'baseline_tvt_all_slope': bl_all, 'baseline_tvt_recent_slope': bl_recent,
        'GR_minus_last_known': gr_filled.to_numpy(float)-ps_gr,
        'z_slope_10': zs10, 'z_slope_50': zs50, 'md_cumsum_from_ps': mc_ps,
    }

    for w in [11, 51, 151]:
        r = gr_filled.rolling(w, center=True, min_periods=max(2, w//5))
        fd[f'GR_roll_mean_{w}'] = r.mean().bfill().ffill().to_numpy(float)
        fd[f'GR_roll_std_{w}']  = r.std().bfill().ffill().fillna(0).to_numpy(float)
    fd['GR_roll_p10_51']   = rolling_quantile(gr_filled, 51, 0.10)
    fd['GR_roll_p90_51']   = rolling_quantile(gr_filled, 51, 0.90)
    fd['GR_roll_range_51'] = fd['GR_roll_p90_51'] - fd['GR_roll_p10_51']
    fd['GR_diff_1']  = gr_filled.diff(1).fillna(0).to_numpy(float)
    fd['GR_diff_10'] = gr_filled.diff(10).fillna(0).to_numpy(float)
    fd['GR_diff_50'] = gr_filled.diff(50).fillna(0).to_numpy(float)

    if {'TVT','GR'}.issubset(tw.columns) and len(tw)>1:
        tw_tvt = tw['TVT'].astype(float); tw_gr = tw['GR'].astype(float)
        tw_at_base = safe_interp(baseline, tw_tvt, tw_gr)
        tw_at_last = safe_interp(np.array([last_known_tvt]), tw_tvt, tw_gr)[0]
        fd['typewell_tvt_min']        = np.full(n, tw_tvt.min())
        fd['typewell_tvt_max']        = np.full(n, tw_tvt.max())
        fd['typewell_tvt_range']      = np.full(n, tw_tvt.max()-tw_tvt.min())
        fd['typewell_gr_mean']        = np.full(n, tw_gr.mean())
        fd['typewell_gr_std']         = np.full(n, tw_gr.std())
        fd['tw_gr_at_baseline_tvt']   = tw_at_base
        fd['tw_gr_at_last_known_tvt'] = np.full(n, tw_at_last)
        fd['GR_minus_tw_baseline']    = gr_filled.to_numpy(float) - tw_at_base
        fd['GR_minus_tw_last_known']  = gr_filled.to_numpy(float) - tw_at_last
    else:
        for col in ['typewell_tvt_min','typewell_tvt_max','typewell_tvt_range','typewell_gr_mean',
                    'typewell_gr_std','tw_gr_at_baseline_tvt','tw_gr_at_last_known_tvt',
                    'GR_minus_tw_baseline','GR_minus_tw_last_known']:
            fd[col] = np.full(n, np.nan)

    feature_names = list(fd)
    X_full   = np.vstack([fd[c] for c in feature_names]).T.astype(np.float32)
    baseline = baseline.astype(np.float32)

    y_tvt = h['TVT'].astype(float).to_numpy(dtype=np.float32) if split=='train' else np.full(n, np.nan, np.float32)
    y_res = (y_tvt - baseline).astype(np.float32) if split=='train' else np.full(n, np.nan, np.float32)

    return dict(well_id=well_id, feature_names=feature_names, X=X_full,
                baseline=baseline, y_tvt=y_tvt, y_residual=y_res,
                target_mask=target_mask.astype(bool), row_index=row_index)


## 4. Build All Training Data (once, shared)

In [ ]:

print('Building LightGBM features...')
lgb_parts = []
for i, path in enumerate(train_horizontal_paths, 1):
    feat = build_lgb_features(path, split='train')
    if feat is not None: lgb_parts.append(feat)
    if i % 100 == 0: print(f'  LGB {i}/{len(train_horizontal_paths)}')

lgb_train_df = pd.concat(lgb_parts, ignore_index=True)
del lgb_parts; gc.collect()
print(f'LGB train table: {lgb_train_df.shape}')


In [ ]:

print('Building XGB/NN sequences...')
train_sequences = []
feature_names   = None
for i, path in enumerate(train_horizontal_paths, 1):
    seq = build_sequence(path, split='train')
    if feature_names is None: feature_names = seq['feature_names']
    train_sequences.append(seq)
    if i % 100 == 0: print(f'  SEQ {i}/{len(train_horizontal_paths)}')

print(f'Sequences: {len(train_sequences)} | Features: {len(feature_names)}')
print(f'Total target rows: {sum(int(s["target_mask"].sum()) for s in train_sequences)}')


## 5. LightGBM — GroupKFold Training

In [ ]:
LGB_EXCLUDE = {'well_id', 'row_index', 'target_tvt', 'target_residual'}
lgb_feat_cols = [c for c in lgb_train_df.columns if c not in LGB_EXCLUDE]

X_lgb    = lgb_train_df[lgb_feat_cols].astype(np.float32)
y_lgb    = lgb_train_df['target_residual'].astype(np.float32)  # residual over last_known_tvt
ytrue_lgb = lgb_train_df['target_tvt'].astype(np.float32)
base_lgb  = lgb_train_df['last_known_tvt'].astype(np.float32)
grp_lgb   = lgb_train_df['well_id'].values

gkf = GroupKFold(n_splits=N_FOLDS)
lgb_oof = np.zeros(len(lgb_train_df), dtype=np.float32)
lgb_models = []
lgb_fold_rmses = []

for fold, (trn, val) in enumerate(gkf.split(X_lgb, y_lgb, grp_lgb), 1):
    print(f'\n===== LGB Fold {fold}/{N_FOLDS} =====')
    model = LGBMRegressor(**LGB_PARAMS)
    model.fit(X_lgb.iloc[trn], y_lgb.iloc[trn],
              eval_set=[(X_lgb.iloc[val], y_lgb.iloc[val])],
              callbacks=[__import__('lightgbm').early_stopping(50, verbose=False),
                         __import__('lightgbm').log_evaluation(100)])
    val_pred = base_lgb.iloc[val].to_numpy() + model.predict(X_lgb.iloc[val]).astype(np.float32)
    lgb_oof[val] = val_pred
    fr = rmse(ytrue_lgb.iloc[val], val_pred)
    br = rmse(ytrue_lgb.iloc[val], base_lgb.iloc[val])
    lgb_fold_rmses.append(fr)
    lgb_models.append(model)
    print(f'Baseline: {br:.5f} | LGB: {fr:.5f}')

lgb_oof_rmse = rmse(ytrue_lgb, lgb_oof)
print(f'\nLGB OOF RMSE: {lgb_oof_rmse:.5f} | Mean fold: {np.mean(lgb_fold_rmses):.5f}')


## 6. XGBoost — GroupKFold Training

In [ ]:

xgb_rows = []
for seq in train_sequences:
    m = seq['target_mask']
    if m.sum() == 0: continue
    df = pd.DataFrame(seq['X'][m], columns=feature_names)
    df['well_id'] = seq['well_id']
    df['target_tvt'] = seq['y_tvt'][m]
    df['target_residual'] = seq['y_residual'][m]
    df['baseline_tvt'] = seq['baseline'][m]
    df['row_index_orig'] = seq['row_index'][m]
    xgb_rows.append(df)

xgb_train_df = pd.concat(xgb_rows, ignore_index=True)
del xgb_rows; gc.collect()

XGB_EXCLUDE  = {'well_id','target_tvt','target_residual','baseline_tvt','row_index_orig'}
xgb_feat_cols = [c for c in xgb_train_df.columns if c not in XGB_EXCLUDE]

X_xgb    = xgb_train_df[xgb_feat_cols].astype(np.float32)
y_xgb    = xgb_train_df['target_residual'].astype(np.float32)
ytrue_xgb = xgb_train_df['target_tvt'].astype(np.float32)
base_xgb  = xgb_train_df['baseline_tvt'].astype(np.float32)
grp_xgb   = xgb_train_df['well_id'].values

xgb_oof = np.zeros(len(xgb_train_df), dtype=np.float32)
xgb_models = []
xgb_fold_rmses = []

for fold, (trn, val) in enumerate(gkf.split(X_xgb, y_xgb, grp_xgb), 1):
    print(f'\n===== XGB Fold {fold}/{N_FOLDS} =====')
    model = XGBRegressor(**XGB_PARAMS)
    model.fit(X_xgb.iloc[trn], y_xgb.iloc[trn],
              eval_set=[(X_xgb.iloc[val], y_xgb.iloc[val])], verbose=100)
    val_pred = base_xgb.iloc[val].to_numpy() + model.predict(X_xgb.iloc[val]).astype(np.float32)
    xgb_oof[val] = val_pred
    fr = rmse(ytrue_xgb.iloc[val], val_pred)
    br = rmse(ytrue_xgb.iloc[val], base_xgb.iloc[val])
    xgb_fold_rmses.append(fr)
    xgb_models.append(model)
    print(f'Baseline: {br:.5f} | XGB: {fr:.5f}')

xgb_oof_rmse = rmse(ytrue_xgb, xgb_oof)
print(f'\nXGB OOF RMSE: {xgb_oof_rmse:.5f} | Mean fold: {np.mean(xgb_fold_rmses):.5f}')


## 7. Neural Network — GroupKFold Training

In [ ]:
def compute_feature_scaler(seqs):
    F = seqs[0]['X'].shape[1]
    s, s2, c = np.zeros(F,np.float64), np.zeros(F,np.float64), np.zeros(F,np.float64)
    for seq in seqs:
        X = seq['X'].astype(np.float64); m = np.isfinite(X); X0 = np.where(m,X,0.)
        s += X0.sum(0); s2 += (X0**2).sum(0); c += m.sum(0)
    c = np.maximum(c,1.); mean = s/c
    return mean.astype(np.float32), np.sqrt(np.maximum(s2/c-mean**2,1e-6)).astype(np.float32)

def compute_target_scaler(seqs):
    v = np.concatenate([s['y_residual'][s['target_mask']] for s in seqs])
    return float(np.nanmean(v)), float(np.nanstd(v)+1e-6)

def compute_residual_clip(seqs):
    v = np.concatenate([np.abs(s['y_residual'][s['target_mask']]) for s in seqs])
    return float(np.nanpercentile(v, 99))

def transform_features(X, mean, std):
    return np.nan_to_num((X-mean)/std, nan=0., posinf=0., neginf=0.).astype(np.float32)

class WellDS(Dataset):
    def __init__(self, seqs, fm, fs, ym, ys):
        self.seqs=seqs; self.fm=fm; self.fs=fs; self.ym=ym; self.ys=ys
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i):
        s=self.seqs[i]
        X=transform_features(s['X'],self.fm,self.fs)
        y=np.nan_to_num((s['y_residual']-self.ym)/self.ys,nan=0.,posinf=0.,neginf=0.).astype(np.float32)
        return dict(X=torch.tensor(X,dtype=torch.float32),
                    y=torch.tensor(y,dtype=torch.float32),
                    tm=torch.tensor(s['target_mask'],dtype=torch.bool))

def collate(batch):
    B=len(batch); L=max(b['X'].shape[0] for b in batch); F=batch[0]['X'].shape[1]
    X=torch.zeros(B,L,F); y=torch.zeros(B,L); tm=torch.zeros(B,L,dtype=torch.bool)
    for i,b in enumerate(batch):
        l=b['X'].shape[0]; X[i,:l]=b['X']; y[i,:l]=b['y']; tm[i,:l]=b['tm']
    return X,y,tm

class ResBlock(nn.Module):
    def __init__(self, ch, ks=5, dil=1, dr=0.10):
        super().__init__()
        p = dil*(ks-1)//2
        self.b = nn.Sequential(
            nn.Conv1d(ch,ch,ks,padding=p,dilation=dil), nn.BatchNorm1d(ch), nn.SiLU(), nn.Dropout(dr),
            nn.Conv1d(ch,ch,ks,padding=p,dilation=dil), nn.BatchNorm1d(ch), nn.SiLU(), nn.Dropout(dr))
        self.fine = nn.Sequential(nn.Conv1d(ch,ch//4,3,padding=1), nn.SiLU(), nn.Conv1d(ch//4,ch,1))
        self.g = nn.Parameter(torch.zeros(1))
    def forward(self,x): return x + self.b(x) + torch.sigmoid(self.g)*self.fine(x)

class TCN(nn.Module):
    def __init__(self, nf, hs=128, nb=6, dr=0.10):
        super().__init__()
        self.inp = nn.Sequential(nn.Conv1d(nf,hs,1), nn.BatchNorm1d(hs), nn.SiLU())
        self.blks = nn.Sequential(*[ResBlock(hs,5,2**i,dr) for i in range(nb)])
        self.head = nn.Sequential(nn.Conv1d(hs,hs//2,1), nn.SiLU(), nn.Conv1d(hs//2,1,1))
    def forward(self,x):
        x=x.transpose(1,2); x=self.inp(x); x=self.blks(x); return self.head(x).squeeze(1)

def huber(pred,tgt,mask,delta=HUBER_DELTA):
    if mask.sum()==0: return None
    return nn.functional.huber_loss(pred[mask],tgt[mask],delta=delta)

def get_lr(e,wu,tot,base):
    if e<wu: return base*(e+1)/max(wu,1)
    p=(e-wu)/max(tot-wu,1); return base*0.5*(1+np.cos(np.pi*p))

def predict_nn(model, seqs, fm, fs, ym, ys, clip=None):
    model.eval(); rows=[]
    with torch.no_grad():
        for seq in seqs:
            X=transform_features(seq['X'],fm,fs)
            pr=model(torch.tensor(X[None],dtype=torch.float32,device=DEVICE)).cpu().numpy()[0]*ys+ym
            if clip: pr=np.clip(pr,-clip,clip)
            pt=seq['baseline']+pr; m=seq['target_mask']
            rows.append(pd.DataFrame({'well_id':seq['well_id'],'row_index':seq['row_index'][m],
                                       'actual_tvt':seq['y_tvt'][m],'pred_tvt':pt[m]}))
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()

def train_nn_fold(fold, tr_seqs, va_seqs):
    fm,fs=compute_feature_scaler(tr_seqs); ym,ys=compute_target_scaler(tr_seqs)
    clip=compute_residual_clip(tr_seqs)
    loader=DataLoader(WellDS(tr_seqs,fm,fs,ym,ys), batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=0, collate_fn=collate, pin_memory=torch.cuda.is_available())
    model=TCN(len(feature_names),HIDDEN_SIZE,NUM_BLOCKS,DROPOUT).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    best_r,best_s=np.inf,None
    for ep in range(EPOCHS):
        lr=get_lr(ep,WARMUP_EPOCHS,EPOCHS,LEARNING_RATE)
        for pg in opt.param_groups: pg['lr']=lr
        model.train(); losses=[]
        for X,y,tm in loader:
            X,y,tm=X.to(DEVICE),y.to(DEVICE),tm.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            l=huber(model(X),y,tm)
            if l is None: continue
            l.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); opt.step()
            losses.append(float(l.detach().cpu()))
        vp=predict_nn(model,va_seqs,fm,fs,ym,ys,clip)
        vr=rmse(vp['actual_tvt'],vp['pred_tvt'])
        print(f'  Fold {fold} ep {ep+1:02d} lr {lr:.5f} loss {np.mean(losses):.4f} NN {vr:.4f}')
        if vr<best_r: best_r=vr; best_s=copy.deepcopy(model.state_dict())
    model.load_state_dict(best_s)
    return model, fm, fs, ym, ys, predict_nn(model,va_seqs,fm,fs,ym,ys,clip), clip


In [ ]:
well_seq_ids = np.array([s['well_id'] for s in train_sequences])
nn_models=[]; nn_scalers=[]; nn_clips=[]; nn_oof_parts=[]; nn_fold_rmses=[]

for fold,(trn,val) in enumerate(GroupKFold(N_FOLDS).split(well_seq_ids,groups=well_seq_ids),1):
    print(f'\n===== NN Fold {fold}/{N_FOLDS} =====')
    model,fm,fs,ym,ys,vp,clip = train_nn_fold(fold,
        [train_sequences[i] for i in trn],
        [train_sequences[i] for i in val])
    fr=rmse(vp['actual_tvt'],vp['pred_tvt'])
    print(f'NN Fold {fold} best RMSE: {fr:.5f}')
    nn_fold_rmses.append(fr); nn_models.append(model)
    nn_scalers.append((fm,fs,ym,ys)); nn_clips.append(clip)
    nn_oof_parts.append(vp.assign(fold=fold))
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

nn_oof_df   = pd.concat(nn_oof_parts, ignore_index=True)
nn_oof_rmse = rmse(nn_oof_df['actual_tvt'], nn_oof_df['pred_tvt'])
print(f'\nNN OOF RMSE: {nn_oof_rmse:.5f} | Mean fold: {np.mean(nn_fold_rmses):.5f}')


## 8. Find Optimal 3-Way Blend Weights on OOF

Grid search over `(w_lgb, w_xgb)` where `w_nn = 1 - w_lgb - w_xgb`.
We only need to align rows that all three models predicted (the intersection).


In [ ]:

lgb_train_df['lgb_oof_tvt'] = lgb_oof
lgb_lookup = {(r['well_id'], int(r['row_index'])): float(r['lgb_oof_tvt'])
              for _, r in lgb_train_df.iterrows()}


xgb_train_df['xgb_oof_tvt'] = xgb_oof
xgb_lookup = {(r['well_id'], int(r['row_index_orig'])): float(r['xgb_oof_tvt'])
              for _, r in xgb_train_df.iterrows()}


nn_lookup = {(r['well_id'], int(r['row_index'])): float(r['pred_tvt'])
             for _, r in nn_oof_df.iterrows()}


align_df = xgb_train_df[['well_id','row_index_orig','target_tvt','xgb_oof_tvt']].copy()
align_df['lgb_oof_tvt'] = [(lgb_lookup.get((w,int(r)), np.nan))
                            for w,r in zip(align_df['well_id'], align_df['row_index_orig'])]
align_df['nn_oof_tvt']  = [(nn_lookup.get((w,int(r)), np.nan))
                            for w,r in zip(align_df['well_id'], align_df['row_index_orig'])]

align_df = align_df.dropna(subset=['lgb_oof_tvt','xgb_oof_tvt','nn_oof_tvt'])
print(f'Aligned rows for 3-way blend search: {len(align_df)}')

y_al   = align_df['target_tvt'].to_numpy()
lgb_al = align_df['lgb_oof_tvt'].to_numpy()
xgb_al = align_df['xgb_oof_tvt'].to_numpy()
nn_al  = align_df['nn_oof_tvt'].to_numpy()

print(f'LGB OOF RMSE: {rmse(y_al,lgb_al):.5f}')
print(f'XGB OOF RMSE: {rmse(y_al,xgb_al):.5f}')
print(f'NN  OOF RMSE: {rmse(y_al,nn_al):.5f}')


step = 0.05
vals = np.arange(0.0, 1.0+step, step)
best_rmse_val = np.inf
best_w = (0.33, 0.33, 0.34)
results = []

for wl in vals:
    for wx in vals:
        wn = 1.0 - wl - wx
        if wn < -1e-6: continue
        wn = max(wn, 0.0)
        blend = wl*lgb_al + wx*xgb_al + wn*nn_al
        r = rmse(y_al, blend)
        results.append((wl, wx, wn, r))
        if r < best_rmse_val:
            best_rmse_val = r; best_w = (wl, wx, wn)

w_lgb, w_xgb, w_nn = best_w
print(f'\nBest weights:  LGB={w_lgb:.2f}  XGB={w_xgb:.2f}  NN={w_nn:.2f}')
print(f'Best blend OOF RMSE: {best_rmse_val:.5f}')


import matplotlib.tri as tri
wls = [r[0] for r in results]; wxs = [r[1] for r in results]; rs = [r[3] for r in results]
fig, ax = plt.subplots(figsize=(6,5))
sc = ax.scatter(wls, wxs, c=rs, cmap='viridis_r', s=30)
ax.scatter([w_lgb],[w_xgb], marker='*', s=200, color='red', zorder=5, label=f'best ({w_lgb:.2f},{w_xgb:.2f})')
plt.colorbar(sc, ax=ax, label='OOF RMSE')
ax.set_xlabel('w_lgb'); ax.set_ylabel('w_xgb')
ax.set_title('3-way blend weight grid search'); ax.legend()
plt.tight_layout(); plt.show()


## 9. Test Predictions

In [ ]:

print('Building LGB test features...')
lgb_test_parts = []
for i,path in enumerate(test_horizontal_paths,1):
    feat = build_lgb_features(path, split='test')
    if feat is not None: lgb_test_parts.append(feat)
    if i % 50 == 0: print(f'  LGB test {i}/{len(test_horizontal_paths)}')

lgb_test_df = pd.concat(lgb_test_parts, ignore_index=True)
del lgb_test_parts; gc.collect()

lgb_test_resid = np.zeros(len(lgb_test_df), dtype=np.float32)
for fold,model in enumerate(lgb_models,1):
    lgb_test_resid += model.predict(lgb_test_df[lgb_feat_cols].astype(np.float32)).astype(np.float32) / len(lgb_models)
    print(f'LGB fold {fold} predicted')

lgb_test_df['lgb_tvt'] = lgb_test_df['last_known_tvt'].to_numpy() + lgb_test_resid
lgb_map = {(r['well_id'], int(r['row_index'])): float(r['lgb_tvt']) for _,r in lgb_test_df.iterrows()}
print(f'LGB test preds: {len(lgb_map)}')


In [ ]:

print('Building XGB/NN test sequences...')
test_sequences = []
for i,path in enumerate(test_horizontal_paths,1):
    test_sequences.append(build_sequence(path, split='test'))
    if i % 50 == 0: print(f'  SEQ test {i}/{len(test_horizontal_paths)}')
print(f'Test sequences: {len(test_sequences)}')


In [ ]:

test_xgb_rows = []
for seq in test_sequences:
    m = seq['target_mask']
    if m.sum()==0: continue
    df = pd.DataFrame(seq['X'][m], columns=feature_names)
    df['well_id']=seq['well_id']; df['row_index']=seq['row_index'][m]
    df['baseline_tvt']=seq['baseline'][m]
    test_xgb_rows.append(df)

test_df_xgb = pd.concat(test_xgb_rows, ignore_index=True)
del test_xgb_rows; gc.collect()

X_test_xgb = test_df_xgb[xgb_feat_cols].astype(np.float32)
xgb_test_resid = np.zeros(len(test_df_xgb), np.float32)
for fold,model in enumerate(xgb_models,1):
    xgb_test_resid += model.predict(X_test_xgb).astype(np.float32)/len(xgb_models)
    print(f'XGB fold {fold} predicted')

test_df_xgb['xgb_tvt'] = test_df_xgb['baseline_tvt'].to_numpy() + xgb_test_resid
xgb_map = {(r['well_id'],int(r['row_index'])):float(r['xgb_tvt']) for _,r in test_df_xgb.iterrows()}
print(f'XGB test preds: {len(xgb_map)}')


In [ ]:

GR_IDX = feature_names.index('GR_filled') if 'GR_filled' in feature_names else None
nn_w_raw = 1.0 / np.maximum(np.array(nn_fold_rmses), 1e-6)
nn_weights = nn_w_raw / nn_w_raw.sum()
print('NN fold weights:', np.round(nn_weights, 4))

nn_sum = defaultdict(float); nn_cnt = defaultdict(float)
for fi,(model,scaler,clip,w) in enumerate(zip(nn_models,nn_scalers,nn_clips,nn_weights),1):
    fm,fs,ym,ys = scaler
    model.eval()
    with torch.no_grad():
        for seq in test_sequences:
            bX = transform_features(seq['X'],fm,fs); m = seq['target_mask']
            preds=[]
            for t in range(TTA_ROUNDS):
                Xa=bX.copy()
                if t>0 and GR_IDX:
                    sc=np.abs(bX[:,GR_IDX]).mean()+1e-6
                    Xa[:,GR_IDX]+=np.random.normal(0,TTA_GR_NOISE*sc,Xa.shape[0]).astype(np.float32)
                xt=torch.tensor(Xa[None],dtype=torch.float32,device=DEVICE)
                pr=model(xt).cpu().numpy()[0]*ys+ym
                preds.append(seq['baseline']+np.clip(pr,-clip,clip))
            avg=np.mean(preds,0)
            for ri,tv in zip(seq['row_index'][m],avg[m]):
                key=(seq['well_id'],int(ri))
                nn_sum[key]+=w*float(tv); nn_cnt[key]+=w
    print(f'NN fold {fi} predicted (w={w:.4f})')

nn_map = {k: nn_sum[k]/nn_cnt[k] for k in nn_sum}
print(f'NN test preds: {len(nn_map)}')


## 10. Blend & Save Submission

In [ ]:

test_df_xgb['id'] = test_df_xgb['well_id'] + '_' + test_df_xgb['row_index'].astype(int).astype(str)
test_df_xgb['lgb_tvt'] = [(lgb_map.get((r['well_id'],int(r['row_index'])), np.nan))
                           for _,r in test_df_xgb.iterrows()]
test_df_xgb['nn_tvt']  = [(nn_map.get((r['well_id'],int(r['row_index'])), np.nan))
                           for _,r in test_df_xgb.iterrows()]


test_df_xgb['lgb_tvt'] = test_df_xgb['lgb_tvt'].fillna(test_df_xgb['xgb_tvt'])
test_df_xgb['nn_tvt']  = test_df_xgb['nn_tvt'].fillna(test_df_xgb['xgb_tvt'])

test_df_xgb['tvt'] = (w_lgb * test_df_xgb['lgb_tvt']
                    + w_xgb * test_df_xgb['xgb_tvt']
                    + w_nn  * test_df_xgb['nn_tvt'])

print(f'Final blend:  LGB={w_lgb:.2f}  XGB={w_xgb:.2f}  NN={w_nn:.2f}')
display(test_df_xgb[['id','baseline_tvt','lgb_tvt','xgb_tvt','nn_tvt','tvt']].head(10))

submission = test_df_xgb[['id','tvt']].copy()
submission['tvt'] = pd.to_numeric(submission['tvt'], errors='coerce')
submission.to_csv('submission.csv', index=False)
print(f'Saved submission.csv — {len(submission):,} rows')
display(submission['tvt'].describe())


In [ ]:

print('='*55)
print(f'LGB OOF RMSE:          {rmse(y_al,lgb_al):.5f}')
print(f'XGB OOF RMSE:          {rmse(y_al,xgb_al):.5f}')
print(f'NN  OOF RMSE:          {rmse(y_al,nn_al):.5f}')
print('-'*55)
equal_blend = rmse(y_al, (lgb_al+xgb_al+nn_al)/3)
print(f'Equal-weight blend:    {equal_blend:.5f}')
print(f'Optimal blend OOF:     {best_rmse_val:.5f}  (LGB={w_lgb:.2f} XGB={w_xgb:.2f} NN={w_nn:.2f})')
print('='*55)


**Acknowledgement :**


1. [https://www.kaggle.com/code/cdeotte/xgb-starter-cv-15](https://www.kaggle.com/code/cdeotte/xgb-starter-cv-15)
2. [https://www.kaggle.com/code/cdeotte/nn-starter-cv-15-5](https://www.kaggle.com/code/cdeotte/nn-starter-cv-15-5)
3. [https://www.kaggle.com/code/vishwasmishra1234/rogii-wellbore-geology-prediction](https://www.kaggle.com/code/vishwasmishra1234/rogii-wellbore-geology-prediction)